In [0]:
dbutils.library.restartPython()

In [0]:
import sys

sys.path.append("/Workspace/Users/sjaddya98@gmail.com/energy-lakehouse/src")

In [0]:
from pyspark.sql.types import StringType, StructField, StructType

from energy_lakehouse.silver import night_hours_sql, station_lookup

print("session timezone was:", spark.conf.get("spark.sql.session.timeZone"))
spark.conf.set("spark.sql.session.timeZone", "UTC")

STATION_SCHEMA = StructType(
    [
        StructField("station_id", StringType(), False),
        StructField("station_name", StringType(), False),
        StructField("state", StringType(), False),
    ]
)

spark.createDataFrame(station_lookup(), STATION_SCHEMA).createOrReplaceTempView("station_lookup")
spark.table("station_lookup").show(truncate=False)
print("night hours:", night_hours_sql())

In [0]:
from energy_lakehouse.silver import night_hours_sql

NIGHT = night_hours_sql()

spark.sql(f"""
CREATE OR REPLACE TABLE workspace.silver.weather_station_hourly AS
WITH shifted AS (
    SELECT station_id,
           timestamp_utc - INTERVAL 1 HOUR AS hour_start_utc,
           parameter,
           value,
           quality
    FROM workspace.bronze.dwd_raw
),
wide AS (
    SELECT station_id,
           hour_start_utc,
           max(CASE WHEN parameter = 'wind_speed_ms'      THEN value   END) AS wind_speed_ms,
           max(CASE WHEN parameter = 'wind_direction_deg' THEN value   END) AS wind_direction_deg,
           max(CASE WHEN parameter = 'sunshine_minutes'   THEN value   END) AS sunshine_minutes,
           max(CASE WHEN parameter = 'wind_speed_ms'      THEN quality END) AS wind_quality,
           max(CASE WHEN parameter = 'sunshine_minutes'   THEN quality END) AS sun_quality
    FROM shifted
    GROUP BY station_id, hour_start_utc
),
bounds AS (
    SELECT min(hour_start_utc) AS first_hour,
           max(hour_start_utc) AS last_hour
    FROM wide
),
spine AS (
    SELECT lk.station_id,
           lk.station_name,
           lk.state,
           explode(sequence(b.first_hour, b.last_hour, INTERVAL 1 HOUR)) AS hour_start_utc
    FROM station_lookup AS lk
    CROSS JOIN bounds AS b
),
joined AS (
    SELECT sp.hour_start_utc,
           sp.station_id,
           sp.station_name,
           sp.state,
           hour(sp.hour_start_utc) IN {NIGHT} AS is_night_hour,
           w.wind_speed_ms,
           w.wind_direction_deg,
           w.sunshine_minutes AS sunshine_minutes_raw,
           w.wind_quality,
           w.sun_quality
    FROM spine AS sp
    LEFT JOIN wide AS w
           ON w.station_id = sp.station_id
          AND w.hour_start_utc = sp.hour_start_utc
)
SELECT hour_start_utc,
       from_utc_timestamp(hour_start_utc, 'Europe/Berlin')       AS hour_start_berlin,
       date(from_utc_timestamp(hour_start_utc, 'Europe/Berlin')) AS date_berlin,
       hour(from_utc_timestamp(hour_start_utc, 'Europe/Berlin')) AS hour_of_day_berlin,
       station_id,
       station_name,
       state,
       wind_speed_ms,
       wind_direction_deg,
       CASE WHEN sunshine_minutes_raw IS NOT NULL THEN sunshine_minutes_raw
            WHEN is_night_hour                    THEN 0.0
       END AS sunshine_minutes,
       (sunshine_minutes_raw IS NULL AND is_night_hour) AS sun_is_imputed,
       wind_quality,
       sun_quality
FROM joined
""")

print(spark.table("workspace.silver.weather_station_hourly").count(), "rows")

In [0]:
%sql
SELECT count(*)                       AS rows,
       count(DISTINCT station_id)      AS stations,
       count(DISTINCT hour_start_utc)  AS hours,
       min(hour_start_utc)             AS first_hour,
       max(hour_start_utc)             AS last_hour,
       (SELECT count(*) FROM (
          SELECT station_id, hour_start_utc
          FROM workspace.silver.weather_station_hourly
          GROUP BY station_id, hour_start_utc HAVING count(*) > 1)) AS duplicate_keys
FROM workspace.silver.weather_station_hourly

In [0]:
%sql
SELECT (SELECT max(timestamp_utc)  FROM workspace.bronze.dwd_raw)                  AS bronze_last_hour,
       (SELECT max(hour_start_utc) FROM workspace.silver.weather_station_hourly)   AS silver_last_hour

In [0]:
%sql
SELECT station_id, station_name,
       count(*)                                      AS rows,
       count(sunshine_minutes)                       AS sunshine_values,
       sum(CASE WHEN sun_is_imputed THEN 1 ELSE 0 END) AS imputed_hours,
       count(wind_speed_ms)                          AS wind_values
FROM workspace.silver.weather_station_hourly
GROUP BY station_id, station_name
ORDER BY station_id

In [0]:
%sql
SELECT count(*)                                        AS spine_rows,
       count(sunshine_minutes)                         AS sunshine_not_null,
       sum(CASE WHEN sun_is_imputed THEN 1 ELSE 0 END)   AS imputed,
       count(*) - count(sunshine_minutes)              AS sunshine_null,
       count(sunshine_minutes) - sum(CASE WHEN sun_is_imputed THEN 1 ELSE 0 END) AS measured
FROM workspace.silver.weather_station_hourly

In [0]:
%sql
SELECT date_berlin, count(DISTINCT hour_start_utc) AS hours
FROM workspace.silver.weather_station_hourly
WHERE station_id = '00183'
  AND date_berlin IN (DATE '2026-03-28', DATE '2026-03-29', DATE '2026-03-30',
                      DATE '2026-10-24', DATE '2026-10-25')
GROUP BY date_berlin
ORDER BY date_berlin

In [0]:
%sql
SELECT min(wind_speed_ms) AS min_wind, max(wind_speed_ms) AS max_wind,
       min(wind_direction_deg) AS min_dir, max(wind_direction_deg) AS max_dir,
       min(sunshine_minutes) AS min_sun, max(sunshine_minutes) AS max_sun,
       count(*) - count(wind_speed_ms) AS wind_nulls
FROM workspace.silver.weather_station_hourly

In [0]:
from energy_lakehouse.silver import MIN_STATIONS

spark.sql(f"""
CREATE OR REPLACE TABLE workspace.silver.weather_germany_hourly AS
SELECT hour_start_utc,
       from_utc_timestamp(hour_start_utc, 'Europe/Berlin')       AS hour_start_berlin,
       date(from_utc_timestamp(hour_start_utc, 'Europe/Berlin')) AS date_berlin,
       hour(from_utc_timestamp(hour_start_utc, 'Europe/Berlin')) AS hour_of_day_berlin,

       avg(wind_speed_ms)                                        AS wind_speed_ms_avg,
       count(wind_speed_ms)                                      AS wind_stations,
       count(wind_speed_ms) >= {MIN_STATIONS}                    AS wind_is_reliable,

       mod(degrees(atan2(avg(sin(radians(wind_direction_deg))),
                         avg(cos(radians(wind_direction_deg))))) + 360, 360)
                                                                 AS wind_direction_deg_avg,

       avg(sunshine_minutes)                                     AS sunshine_minutes_avg,
       count(sunshine_minutes)                                   AS sunshine_stations,
       count(sunshine_minutes) >= {MIN_STATIONS}                 AS sunshine_is_reliable,
       sum(CASE WHEN sun_is_imputed THEN 1 ELSE 0 END)           AS sunshine_imputed_stations
FROM workspace.silver.weather_station_hourly
GROUP BY hour_start_utc
""")

print(spark.table("workspace.silver.weather_germany_hourly").count(), "rows")

In [0]:
%sql
SELECT count(*) AS rows,
       min(hour_start_utc) AS first_hour,
       max(hour_start_utc) AS last_hour,
       sum(CASE WHEN wind_is_reliable      THEN 0 ELSE 1 END) AS wind_unreliable_hours,
       sum(CASE WHEN sunshine_is_reliable  THEN 0 ELSE 1 END) AS sun_unreliable_hours,
       min(wind_stations) AS min_wind_stations,
       min(sunshine_stations) AS min_sun_stations,
       (SELECT count(*) FROM (
          SELECT hour_start_utc FROM workspace.silver.weather_germany_hourly
          GROUP BY hour_start_utc HAVING count(*) > 1)) AS duplicate_keys
FROM workspace.silver.weather_germany_hourly

In [0]:
%sql
SELECT s.station_id, s.wind_speed_ms, s.sunshine_minutes, s.sun_is_imputed
FROM workspace.silver.weather_station_hourly AS s
WHERE s.hour_start_utc = TIMESTAMP '2026-11-11 12:00:00'
ORDER BY s.station_id

In [0]:
%sql
SELECT wind_speed_ms_avg, wind_stations, sunshine_minutes_avg, sunshine_stations,
       sunshine_imputed_stations, wind_direction_deg_avg
FROM workspace.silver.weather_germany_hourly
WHERE hour_start_utc = TIMESTAMP '2026-11-11 12:00:00'

In [0]:
%sql
SELECT wind_stations, sunshine_minutes_avg, sunshine_stations, sunshine_imputed_stations
FROM workspace.silver.weather_germany_hourly
WHERE hour_start_utc = TIMESTAMP '2026-01-15 23:00:00'